In [6]:
class Transaccion:

    def __init__(self, tipo, monto, saldo_resultante):
        self.__tipo = tipo
        self.__monto = monto
        self.__saldo_resultante = saldo_resultante

    def __str__(self):
        return (f"{self.__tipo:<24} S/ {self.__monto:>10.2f}"
                f" Saldo: S/ {self.__saldo_resultante:>10.2f}")
    
    def get_tipo(self):
        return self.__tipo

    def get_monto(self):
        return self.__monto

    def get_saldo_resultante(self):
        return self.__saldo_resultante


### CLASE CLIENTE

In [ ]:
class Cliente:
    def __init__(self, dni, nombre, apellido, telefono=""):
        self.__dni = dni
        self.__nombre = nombre
        self.__apellido = apellido
        self.__telefono = telefono
        self.__cuentas = []

# ---------------- GETTERS ----------------
    def get_dni(self):
        return self.__dni

    def get_nombre_completo(self):
        return self.__nombre + " " + self.__apellido

    def get_telefono(self):
        return self.__telefono

    def get_cuentas(self):
        return self.__cuentas
    
# ---------------- SETTERS ----------------
    def set_telefono(self, nuevo_telefono):
        # Validacion: un celular en Peru tiene 9 digitos
        if nuevo_telefono.isdigit() and len(nuevo_telefono) == 9:
            self.__telefono = nuevo_telefono
        else:
            print("Telefono no valido: debe tener 9 digitos.")

# ---------------- METODOS ----------------
    def agregar_cuenta(self, cuenta):
        self.__cuentas.append(cuenta)

    def saldo_total(self):
        total = 0
        for cuenta in self.__cuentas:
            total += cuenta.get_saldo()
        return total

    def mostrar(self):
        print(f"\nCliente: {self.get_nombre_completo()}  |  DNI: {self.__dni}"
              f"  |  Tel: {self.__telefono}")
        if len(self.__cuentas) == 0:
            print("   (no tiene cuentas)")
        for cuenta in self.__cuentas:
            print("  ", cuenta)
        print(f"   Saldo total: S/ {self.saldo_total():.2f}")

    def __str__(self):
        return f"{self.__dni} - {self.get_nombre_completo()} ({len(self.__cuentas)} cuenta/s)"

    

### CLASE BANCO

In [7]:
class Cuenta:

    contador = 1
    def __init__(self, titular, saldo_inicial=0):
        self.__numero = "001-" + str(Cuenta.contador).zfill(5)
        Cuenta.contador += 1
        self.__titular = titular
        self.__saldo = 0
        self.__movimientos = []
        if saldo_inicial > 0:
            self.depositar(saldo_inicial, "Apertura de cuenta")

    def __str__(self):
        return (f"{self.__numero}  {self.get_tipo():<17} "
                f" Saldo: S/ {self.__saldo:>10.2f}")

    def get_numero(self):
        return self.__numero

    def get_titular(self):
        return self.__titular

    def get_saldo(self):
        return self.__saldo

    def get_movimientos(self):
        return self.__movimientos

    def get_tipo(self):
        return "Cuenta"

    def __registrar(self, tipo, monto):
        self.__movimientos.append(Transaccion(tipo, monto, self.__saldo))
    
    def depositar(self, monto, tipo="Deposito"):
        if monto <= 0:
            print("El monto a depositar debe ser mayor que 0.")
            return 
        self.__saldo += monto
        self.__registrar(tipo, monto)
        return f"Deposito exitoso"

    def puede_retirar(self, monto):
        if monto <= self.__saldo:
            return f"Retiro permitido"
        else:
            print(f"Saldo insuficiente")
            return False

    def retirar(self, monto, tipo="Retiro"):
        if monto <= 0:
            print("El monto a retirar debe ser mayor que 0.")
            return 
        if not self.puede_retirar(monto):
            print(f"Operacion rechazada: no se puede retirar S/ {monto:.2f} "
                  f"de la cuenta {self.__numero}.")
            return
        self.__saldo -= monto
        self.__registrar(tipo, monto)
        return f"Retiro exitoso"
    
    def cierre_de_mes(self):
        pass

    def estado_de_cuenta(self):
        print(f"\n===== ESTADO DE CUENTA {self.__numero} =====")
        print(f"Tipo    : {self.get_tipo()}")
        print(f"Titular : {self.__titular.get_nombre_completo()}")
        print("-" * 58)
        if len(self.__movimientos) == 0:
            print("  (sin movimientos)")
        for i in range(len(self.__movimientos)):
            print(f"  {i + 1:>2}. {self.__movimientos[i]}")
        print("-" * 58)
        print(f"  SALDO ACTUAL: S/ {self.__saldo:.2f}")

In [5]:
class Banco:
    def __init__(self, nombre):
        self.__nombre = nombre
        self.__clientes = []     # LISTA de objetos Cliente
        self.__cuentas = []      # LISTA de objetos Cuenta

    def get_nombre(self):
        return self.__nombre

    def get_clientes(self):
        return self.__clientes

# ---------------- CLIENTES ----------------
    def buscar_cliente(self, dni):
        for cliente in self.__clientes:
            if cliente.get_dni() == dni:
                return cliente
        return None

    def registrar_cliente(self, dni, nombre, apellido, telefono=""):
        # Validaciones
        if not (dni.isdigit() and len(dni) == 8):
            print("DNI no valido: debe tener 8 digitos.")
            return None
        if self.buscar_cliente(dni) is not None:
            print("Ya existe un cliente con ese DNI.")
            return None
        cliente = Cliente(dni, nombre, apellido, telefono)
        self.__clientes.append(cliente)
        print(f"Cliente {cliente.get_nombre_completo()} registrado correctamente.")
        return cliente

# ---------------- CUENTAS ----------------
    def abrir_cuenta(self, dni, tipo, saldo_inicial=0):
        """tipo = 1 (Ahorros) o 2 (Corriente)"""
        cliente = self.buscar_cliente(dni)
        if cliente is None:
            print("No existe un cliente con ese DNI.")
            return None
        if tipo == 1:
            cuenta = CuentaAhorros(cliente, saldo_inicial)
        elif tipo == 2:
            cuenta = CuentaCorriente(cliente, saldo_inicial)
        else:
            print("Tipo de cuenta no valido.")
            return None
        cliente.agregar_cuenta(cuenta)
        self.__cuentas.append(cuenta)
        print(f"Se abrio la {cuenta.get_tipo()} N° {cuenta.get_numero()} "
              f"para {cliente.get_nombre_completo()}.")
        return cuenta

    def buscar_cuenta(self, numero):
        for cuenta in self.__cuentas:
            if cuenta.get_numero() == numero:
                return cuenta
        return None
    
# ---------------- OPERACIONES ----------------
    def transferir(self, numero_origen, numero_destino, monto):
        origen = self.buscar_cuenta(numero_origen)
        destino = self.buscar_cuenta(numero_destino)
        if origen is None or destino is None:
            print("Una de las cuentas no existe.")
            return False
        if origen == destino:
            print("No se puede transferir a la misma cuenta.")
            return False
        # Primero se intenta retirar; solo si funciona se deposita en el destino
        if origen.retirar(monto, "Transferencia enviada"):
            destino.depositar(monto, "Transferencia recibida")
            print(f"Transferencia exitosa de S/ {monto:.2f}: "
                  f"{origen.get_numero()} -> {destino.get_numero()}")
            return True
        return False

    def cierre_de_mes(self):
        print("\n=== CIERRE DE MES ===")
        for cuenta in self.__cuentas:
            saldo_antes = cuenta.get_saldo()
            cuenta.cierre_de_mes()          # POLIMORFISMO: cada cuenta hace lo suyo
            diferencia = cuenta.get_saldo() - saldo_antes
            print(f"  {cuenta.get_numero()} {cuenta.get_tipo():<17} "
                  f"cambio: S/ {diferencia:>8.2f}   nuevo saldo: S/ {cuenta.get_saldo():.2f}")
            
# ---------------- REPORTES ----------------
    def mostrar_clientes(self):
        print(f"\n========== CLIENTES DEL {self.__nombre.upper()} ==========")
        if len(self.__clientes) == 0:
            print("No hay clientes registrados.")
        for cliente in self.__clientes:
            cliente.mostrar()

    def reporte_general(self):
        total = 0
        for cuenta in self.__cuentas:
            total += cuenta.get_saldo()
        print(f"\n=== REPORTE GENERAL - {self.__nombre} ===")
        print(f"  Clientes registrados : {len(self.__clientes)}")
        print(f"  Cuentas abiertas     : {len(self.__cuentas)}")
        print(f"  Dinero total en banco: S/ {total:.2f}")

In [ ]:
def mostrar_titulo(banco):
    print("=" * 58)
    print(f"        SISTEMA BANCARIO - {banco.get_nombre().upper()}")
    print("=" * 58)


def leer_monto(mensaje):
    """Pide un monto. Si el usuario escribe algo que no es numero, devuelve -1."""
    texto = input(mensaje)
    if texto.replace(".", "", 1).isdigit():
        return float(texto)
    return -1


def leer_opcion(mensaje):
    texto = input(mensaje)
    if texto.isdigit():
        return int(texto)
    return -1


def cargar_datos_de_ejemplo(banco):
    """Crea clientes y cuentas listos para la demostracion."""
    banco.registrar_cliente("45678912", "Maria", "Quispe Rojas", "987654321")
    banco.registrar_cliente("71234567", "Carlos", "Mendoza Diaz", "912345678")
    banco.registrar_cliente("40112233", "Lucia", "Torres Vega", "956789123")
    banco.abrir_cuenta("45678912", 1, 1500)     # Maria - Ahorros
    banco.abrir_cuenta("45678912", 2, 800)      # Maria - Corriente
    banco.abrir_cuenta("71234567", 1, 3200)     # Carlos - Ahorros
    banco.abrir_cuenta("40112233", 2, 250)      # Lucia - Corriente


def registrar_cliente(banco):
    dni = input("DNI (8 digitos): ")
    nombre = input("Nombres: ")
    apellido = input("Apellidos: ")
    telefono = input("Telefono: ")
    banco.registrar_cliente(dni, nombre, apellido, telefono)


def abrir_cuenta(banco):
    dni = input("DNI del cliente: ")
    print("Tipo de cuenta:  1) Ahorros   2) Corriente")
    tipo = leer_opcion("Elige: ")
    monto = leer_monto("Deposito inicial (S/): ")
    if monto < 0:
        monto = 0
    banco.abrir_cuenta(dni, tipo, monto)


def depositar(banco):
    cuenta = banco.buscar_cuenta(input("Numero de cuenta (ej. 001-00001): "))
    if cuenta is None:
        print("La cuenta no existe.")
        return
    monto = leer_monto("Monto a depositar (S/): ")
    if cuenta.depositar(monto):
        print(f"Deposito exitoso. Nuevo saldo: S/ {cuenta.get_saldo():.2f}")


def retirar(banco):
    cuenta = banco.buscar_cuenta(input("Numero de cuenta: "))
    if cuenta is None:
        print("La cuenta no existe.")
        return
    monto = leer_monto("Monto a retirar (S/): ")
    if cuenta.retirar(monto):
        print(f"Retiro exitoso. Nuevo saldo: S/ {cuenta.get_saldo():.2f}")


def transferir(banco):
    origen = input("Cuenta de origen: ")
    destino = input("Cuenta de destino: ")
    monto = leer_monto("Monto a transferir (S/): ")
    banco.transferir(origen, destino, monto)


def estado_de_cuenta(banco):
    cuenta = banco.buscar_cuenta(input("Numero de cuenta: "))
    if cuenta is None:
        print("La cuenta no existe.")
    else:
        cuenta.estado_de_cuenta()


def main():
    banco = Banco("Banco Nacional del Ahorro")
    mostrar_titulo(banco)

    opcion = -1
    while opcion != 0:
        print("\n------------------ MENU ------------------")
        print(" 1. Cargar datos de ejemplo")
        print(" 2. Registrar cliente")
        print(" 3. Abrir cuenta")
        print(" 4. Depositar")
        print(" 5. Retirar")
        print(" 6. Transferir")
        print(" 7. Ver estado de cuenta (movimientos)")
        print(" 8. Ver clientes y sus cuentas")
        print(" 9. Cierre de mes (intereses y comisiones)")
        print("10. Reporte general del banco")
        print(" 0. Salir")
        opcion = leer_opcion("Elige una opcion: ")

        if opcion == 1:
            cargar_datos_de_ejemplo(banco)
        elif opcion == 2:
            registrar_cliente(banco)
        elif opcion == 3:
            abrir_cuenta(banco)
        elif opcion == 4:
            depositar(banco)
        elif opcion == 5:
            retirar(banco)
        elif opcion == 6:
            transferir(banco)
        elif opcion == 7:
            estado_de_cuenta(banco)
        elif opcion == 8:
            banco.mostrar_clientes()
        elif opcion == 9:
            banco.cierre_de_mes()
        elif opcion == 10:
            banco.reporte_general()
        elif opcion == 0:
            print("Gracias por usar el sistema. Hasta pronto!")
        else:
            print("Opcion no valida, intenta de nuevo.")


# El menu solo arranca cuando ejecutas "python main.py"
if __name__ == "__main__":
    main()
